In [ ]:
import pandas as pd
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from sklearn.manifold import TSNE

# 1. Load your human annotations and lyrics
# Assuming 'human_annotation.csv' contains 'raw_lyrics' and 'human_valence'
df = pd.read_csv('human_annotation.csv')

# Drop rows with missing lyrics or labels just in case
df = df.dropna(subset=['raw_lyrics', 'human_valence'])

# 2. Initialize Model and Tokenizer (Example using the domain-adapted emotion model)
model_name = "j-hartmann/emotion-english-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load base model to get embeddings (hidden states), not the classification head
model = AutoModel.from_pretrained(model_name) 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3. Function to extract embeddings
def get_embedding(text):
    # Truncate to model's max length (typically 512 tokens)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    # Take the mean of the last hidden state for the sentence embedding
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding

# 4. Apply to dataset
print("Extracting embeddings... this may take a few minutes.")
df['embedding'] = df['raw_lyrics'].apply(get_embedding)

# 5. Dimensionality Reduction using t-SNE
print("Running t-SNE...")
import numpy as np
X = np.stack(df['embedding'].values)

tsne = TSNE(n_components=2, random_state=42, perplexity=15) # Adjust perplexity based on dataset size
X_tsne = tsne.fit_transform(X)

df['tsne_1'] = X_tsne[:, 0]
df['tsne_2'] = X_tsne[:, 1]

# 6. Plotting
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='tsne_1', y='tsne_2',
    hue='human_valence', # Color by human-annotated ground truth
    palette='Set2',
    data=df,
    s=100,
    alpha=0.8
)
plt.title('t-SNE Visualization of RoBERTa Embeddings by Human Sentiment')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend(title='Human Valence')
plt.show()